In [1]:
# imports
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
# Real anchor values
annual_sales_gbp = 34_000_000   # £34 million
annual_operating_profit_gbp = 2_200_000  # £2.2 million

# Compute margin
annual_operating_margin = annual_operating_profit_gbp / annual_sales_gbp
annual_operating_margin

0.06470588235294118

In [3]:
annual_summary = pd.DataFrame({
    "fiscal_year_end": ["2025-03-31"],
    "sales_gbp": [annual_sales_gbp],
    "operating_profit_gbp": [annual_operating_profit_gbp],
    "operating_margin": [annual_operating_margin]
})

annual_summary

,fiscal_year_end,sales_gbp,operating_profit_gbp,operating_margin
0,2025-03-31,34000000,2200000,0.064706


In [4]:
# Generate monthly date range for FY 2024-25 (Apr 2024 to Mar 2025)
months = pd.date_range(start="2024-04-01", end="2025-03-31", freq="MS")

# Create base seasonality weights (example pattern)
# Heavier weight in Nov, Dec; lighter in Jan-Feb
season_weights = {
    1: 0.07,  # Jan
    2: 0.07,  # Feb
    3: 0.08,  # Mar
    4: 0.08,  # Apr
    5: 0.08,  # May
    6: 0.08,  # Jun
    7: 0.09,  # Jul
    8: 0.09,  # Aug
    9: 0.09,  # Sep
    10: 0.09, # Oct
    11: 0.10, # Nov
    12: 0.08  # Dec
}

# Normalize weights so that sum across months equals 1
weights = np.array([season_weights[m.month] for m in months])
weights = weights / weights.sum()

# Generate monthly sales by distributing annual total
monthly_sales = weights * annual_sales_gbp

# Add some random noise around ±5% of each month
np.random.seed(42)  # for reproducibility
noise = np.random.normal(loc=1.0, scale=0.05, size=len(monthly_sales))
monthly_sales_noisy = monthly_sales * noise

# Re-scale so total still matches annual_sales_gbp
monthly_sales_noisy *= annual_sales_gbp / monthly_sales_noisy.sum()

# Build DataFrame
monthly_df = pd.DataFrame({
    "month_start": months,
    "sales_gbp": monthly_sales_noisy
})

monthly_df.head()

,month_start,sales_gbp
0,2024-04-01,2.742978e+06
1,2024-05-01,2.658002e+06
2,2024-06-01,2.763182e+06
3,2024-07-01,3.240365e+06
4,2024-08-01,2.975815e+06


In [5]:
# Base margin from annual figure
base_margin = annual_operating_margin  # ~6.47%

# Add small variation ±1.5 percentage points monthly
margin_noise = np.random.normal(loc=0.0, scale=0.015, size=len(monthly_df))
monthly_df["operating_margin"] = np.clip(base_margin + margin_noise, 0, 1)

# Compute operating profit
monthly_df["operating_profit_gbp"] = monthly_df["sales_gbp"] * monthly_df["operating_margin"]

monthly_df.head()

,month_start,sales_gbp,operating_margin,operating_profit_gbp
0,2024-04-01,2.742978e+06,0.068335,187442.257986
1,2024-05-01,2.658002e+06,0.036007,95705.812502
2,2024-06-01,2.763182e+06,0.038832,107300.201387
3,2024-07-01,3.240365e+06,0.056272,182340.445435
4,2024-08-01,2.975815e+06,0.049513,147342.787758


In [6]:
age_groups = ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
gender_categories = ["Male", "Female", "Other"]

# Example overall distribution assumptions (must sum to 1 for each demographic dimension)
age_distribution = {
    "18-24": 0.10,
    "25-34": 0.25,
    "35-44": 0.25,
    "45-54": 0.20,
    "55-64": 0.12,
    "65+":   0.08
}

gender_distribution = {
    "Male":   0.55,
    "Female": 0.43,
    "Other":  0.02
}

In [7]:
# Assume average order value (AOV) in GBP
aov = 250  # adjust as needed

monthly_df["estimated_transactions"] = monthly_df["sales_gbp"] / aov

# Simulate transaction counts as integers
monthly_df["estimated_transactions"] = monthly_df["estimated_transactions"].round().astype(int)

# Now expand each month into demographic rows
rows = []
for _, row in monthly_df.iterrows():
    month = row["month_start"]
    sales = row["sales_gbp"]
    profit = row["operating_profit_gbp"]
    transactions = row["estimated_transactions"]
    
    # Estimate transaction count per age group
    for age, age_pct in age_distribution.items():
        # transactions for this age group
        age_transactions = int(round(transactions * age_pct))
        # distribute within genders
        for gender, gender_pct in gender_distribution.items():
            count = int(round(age_transactions * gender_pct))
            # compute approximate sales share
            sales_share = count / max(transactions, 1)
            rows.append({
                "month_start": month,
                "age_group": age,
                "gender": gender,
                "transactions": count,
                "sales_gbp_estimate": sales * sales_share,
                "operating_profit_gbp_estimate": profit * sales_share
            })

demographic_df = pd.DataFrame(rows)
demographic_df.head(12)

,month_start,age_group,gender,transactions,sales_gbp_estimate,operating_profit_gbp_estimate
0,2024-04-01,18-24,Male,603,150748.781597,10301.465691
1,2024-04-01,18-24,Female,472,117999.046292,8063.502166
2,2024-04-01,18-24,Other,22,5499.955547,375.841203
3,2024-04-01,25-34,Male,1509,377246.950963,25779.289765
4,2024-04-01,25-34,Female,1179,294747.617750,20141.671725
5,2024-04-01,25-34,Other,55,13749.888869,939.603007
6,2024-04-01,35-44,Male,1509,377246.950963,25779.289765
7,2024-04-01,35-44,Female,1179,294747.617750,20141.671725
8,2024-04-01,35-44,Other,55,13749.888869,939.603007
9,2024-04-01,45-54,Male,1207,301747.561174,20620.015074


In [8]:
# Example channel split assumptions
channel_distribution = {
    "Online": 0.20,
    "In-store": 0.80
}

# Add synthetic channel assignment
def assign_channel(df, channel_dist):
    df = df.copy()
    # Use a random but reproducible split
    rng = np.random.default_rng(123)
    channels = list(channel_dist.keys())
    probs = list(channel_dist.values())
    df["channel"] = rng.choice(channels, size=len(df), p=probs)
    return df

demographic_df = assign_channel(demographic_df, channel_distribution)

demographic_df.head()

,month_start,age_group,gender,transactions,sales_gbp_estimate,operating_profit_gbp_estimate,channel
0,2024-04-01,18-24,Male,603,150748.781597,10301.465691,In-store
1,2024-04-01,18-24,Female,472,117999.046292,8063.502166,Online
2,2024-04-01,18-24,Other,22,5499.955547,375.841203,In-store
3,2024-04-01,25-34,Male,1509,377246.950963,25779.289765,Online
4,2024-04-01,25-34,Female,1179,294747.617750,20141.671725,Online


In [9]:
# Example product tiers
product_tiers = ["Entry", "Mid", "Premium"]
product_tier_probs = [0.40, 0.45, 0.15]

rng = np.random.default_rng(456)
demographic_df["product_tier"] = rng.choice(product_tiers, size=len(demographic_df), p=product_tier_probs)

demographic_df.head()

,month_start,age_group,gender,transactions,sales_gbp_estimate,operating_profit_gbp_estimate,channel,product_tier
0,2024-04-01,18-24,Male,603,150748.781597,10301.465691,In-store,Mid
1,2024-04-01,18-24,Female,472,117999.046292,8063.502166,Online,Mid
2,2024-04-01,18-24,Other,22,5499.955547,375.841203,In-store,Entry
3,2024-04-01,25-34,Male,1509,377246.950963,25779.289765,Online,Mid
4,2024-04-01,25-34,Female,1179,294747.617750,20141.671725,Online,Premium


In [10]:
agg_month = demographic_df.groupby("month_start").agg({
    "transactions": "sum",
    "sales_gbp_estimate": "sum",
    "operating_profit_gbp_estimate": "sum"
}).reset_index()

agg_month.head()

,month_start,transactions,sales_gbp_estimate,operating_profit_gbp_estimate
0,2024-04-01,10972,2.742978e+06,187442.257986
1,2024-05-01,10633,2.658252e+06,95714.814177
2,2024-06-01,11052,2.762932e+06,107290.493597
3,2024-07-01,12961,3.240365e+06,182340.445435
4,2024-08-01,11906,2.976565e+06,147379.923637


In [11]:
comparison = monthly_df[["month_start", "sales_gbp", "operating_profit_gbp"]] \
    .merge(agg_month, on="month_start", how="left", suffixes=("_orig", "_demo"))

comparison.head()

,month_start,sales_gbp,operating_profit_gbp,transactions,sales_gbp_estimate,operating_profit_gbp_estimate
0,2024-04-01,2.742978e+06,187442.257986,10972,2.742978e+06,187442.257986
1,2024-05-01,2.658002e+06,95705.812502,10633,2.658252e+06,95714.814177
2,2024-06-01,2.763182e+06,107300.201387,11052,2.762932e+06,107290.493597
3,2024-07-01,3.240365e+06,182340.445435,12961,3.240365e+06,182340.445435
4,2024-08-01,2.975815e+06,147342.787758,11906,2.976565e+06,147379.923637


In [12]:
annual_demo = agg_month[["sales_gbp_estimate", "operating_profit_gbp_estimate"]].sum()
annual_demo

sales_gbp_estimate               3.400225e+07
operating_profit_gbp_estimate    1.881721e+06
dtype: float64

In [13]:
monthly_df.to_csv("citizen_uk_monthly_summary.csv", index=False)
demographic_df.to_csv("citizen_uk_monthly_demographics.csv", index=False)

print("Files saved: citizen_uk_monthly_summary.csv, citizen_uk_monthly_demographics.csv")

Files saved: citizen_uk_monthly_summary.csv, citizen_uk_monthly_demographics.csv
